# C11-neural-training — Session 5: BatchNorm, Dropout, and Mode Audits

*One 90-minute session. Prerequisites: F5's expectation and variance, C5's
overfitting/regularization vocabulary, C6's parameter inspection, and Session
4's autograd/optimizer lifecycle.*

**Learning contract.** We will derive BatchNorm's forward and backward
identities, distinguish batch statistics from running buffers and affine
parameters, prove inverted-dropout expectation preservation, and audit a
combined model in training and deterministic evaluation modes.


In [ ]:
import torch
from torch import nn

SEED = 20260804
ATOL = 1e-8
RTOL = 1e-6
torch.manual_seed(SEED)
torch.set_default_dtype(torch.float64)
torch.use_deterministic_algorithms(True)

## 1. Batch normalization forward pass

Let $X\in\mathbb R^{N\times D}$: $N$ examples and $D$ features. BatchNorm1d
normalizes each feature column. For feature $d$,

$$\mu_d=\frac1N\sum_i X_{id},\qquad
v_d=\frac1N\sum_i(X_{id}-\mu_d)^2,$$
$$\widehat X_{id}=\frac{X_{id}-\mu_d}{\sqrt{v_d+\varepsilon}},\qquad
Y_{id}=\gamma_d\widehat X_{id}+\beta_d.$$

Shapes: $\mu,v,\gamma,\beta$ are $(D,)$ and broadcast across $N$;
$\widehat X,Y$ are $(N,D)$. Here $\varepsilon>0$ is a fixed numerical-
stability constant. Training normalization uses the biased batch variance
(divisor $N$), matching PyTorch's forward computation.

Without affine scale/shift, each normalized column has mean near zero and
variance $v/(v+\varepsilon)$, near one unless the column is almost constant.

**Checkpoint 1A.** Why does a constant feature column normalize to zeros
rather than divide by zero?

**Checkpoint 1B.** For input shape $(32,10)$, state the shapes of batch mean,
batch variance, $\gamma$, and output.


In [ ]:
X_demo = torch.tensor([
    [1.0, 4.0, 7.0],
    [3.0, 4.0, 9.0],
    [5.0, 4.0, 11.0],
    [7.0, 4.0, 13.0],
])
eps = 1e-5
mu = X_demo.mean(dim=0)
variance = X_demo.var(dim=0, unbiased=False)
x_hat = (X_demo - mu) / torch.sqrt(variance + eps)
print("mean:", mu, "variance:", variance)
print("normalized column means:", x_hat.mean(dim=0))
assert torch.allclose(x_hat.mean(dim=0), torch.zeros(3), atol=ATOL, rtol=RTOL)
assert torch.allclose(x_hat[:, 1], torch.zeros(4), atol=ATOL, rtol=RTOL)

## 2. BatchNorm backward identity

Let $G=\partial L/\partial Y$ with shape $(N,D)$ and
$s=(v+\varepsilon)^{-1/2}$ with shape $(D)$. The affine gradients are

$$d\gamma=\sum_i G_i\odot\widehat X_i,\qquad
d\beta=\sum_i G_i.$$

Combining the centered-value, variance, and mean paths gives

$$\boxed{dX=\frac{\gamma s}{N}\odot
\left(NG-\sum_iG_i-\widehat X\odot
\sum_i(G_i\odot\widehat X_i)\right)}.$$

Sums keep the feature dimension and broadcast across rows. Two useful audits
are $\sum_i dX_i\approx0$ and, when $\varepsilon$ is negligible,
$\sum_i dX_i\odot\widehat X_i\approx0$: normalization removes shift and
scale directions. With nonzero epsilon, the second identity is approximate,
not exact.

**Checkpoint 2A.** Explain why $d\gamma$ and $d\beta$ have shape $(D)$.

**Checkpoint 2B.** Which three dependency paths are combined in $dX$?


In [ ]:
gamma = torch.tensor([1.2, -0.7, 0.5])
beta = torch.tensor([0.1, 0.2, -0.3])
G = torch.tensor([
    [0.2, -0.4, 0.1],
    [-0.1, 0.3, 0.5],
    [0.7, -0.2, -0.6],
    [-0.3, 0.8, 0.2],
])
inv_std = torch.rsqrt(variance + eps)
dgamma = (G * x_hat).sum(dim=0)
dbeta = G.sum(dim=0)
dX_formula = (gamma * inv_std / X_demo.shape[0]) * (
    X_demo.shape[0] * G
    - G.sum(dim=0)
    - x_hat * (G * x_hat).sum(dim=0)
)

X_auto = X_demo.clone().requires_grad_(True)
gamma_auto = gamma.clone().requires_grad_(True)
beta_auto = beta.clone().requires_grad_(True)
mu_auto = X_auto.mean(dim=0)
var_auto = X_auto.var(dim=0, unbiased=False)
y_auto = gamma_auto * (X_auto - mu_auto) / torch.sqrt(var_auto + eps) + beta_auto
(y_auto * G).sum().backward()
assert torch.allclose(X_auto.grad, dX_formula, atol=ATOL, rtol=RTOL)
assert torch.allclose(gamma_auto.grad, dgamma, atol=ATOL, rtol=RTOL)
assert torch.allclose(beta_auto.grad, dbeta, atol=ATOL, rtol=RTOL)
assert torch.allclose(dX_formula.sum(dim=0), torch.zeros(3), atol=ATOL, rtol=RTOL)
print("manual BatchNorm backward matches autograd")

## 3. Parameters, buffers, and running statistics

During training, BatchNorm uses current batch statistics and updates
`running_mean` and `running_var`. These are **buffers**: persistent state
in `state_dict()`, but not trainable parameters and not updated by the
optimizer. With `affine=True`, $\gamma$ (`weight`) and $\beta$ (`bias`)
are parameters.

PyTorch's training forward uses biased batch variance for normalization but
updates `running_var` using the corresponding unbiased estimate. Evaluation
uses the stored running statistics and does not update them. The `momentum`
argument controls the running-stat update convention; it is not optimizer
momentum.

**Checkpoint 3A.** Which four BatchNorm tensors appear in `state_dict()`
when affine parameters and running statistics are enabled?

**Checkpoint 3B.** Why should an optimizer never receive
`running_mean`?


In [ ]:
bn = nn.BatchNorm1d(3, eps=eps, momentum=0.1, affine=True, track_running_stats=True)
print("parameters:", [(name, tuple(value.shape)) for name, value in bn.named_parameters()])
print("buffers:", [(name, tuple(value.shape)) for name, value in bn.named_buffers()])
print("state keys:", list(bn.state_dict()))
assert {name for name, _ in bn.named_parameters()} == {"weight", "bias"}
assert {"running_mean", "running_var", "num_batches_tracked"} <= {
    name for name, _ in bn.named_buffers()
}

## 4. Inverted dropout preserves expectation

Let drop probability be $p$ and keep probability $q=1-p$. For each activation
$x$, draw $M\sim\operatorname{Bernoulli}(q)$ and use

$$Y=\frac{M}{q}x.$$

Then $\mathbb E[Y]=\mathbb E[M]x/q=qx/q=x$. This is **inverted dropout**:
training outputs are scaled, so evaluation can be the identity with no extra
factor. The variance increases during training:
$\operatorname{Var}(Y)=x^2p/q$ for fixed $x$.

The mask has the same shape as the activation tensor. Independent elements are
normally masked independently. A fixed seed can reproduce a sequence of masks,
but successive training calls still consume new draws and need not match.

**Checkpoint 4A.** With $p=1/4$ and input $x=6$, what are the two possible
training outputs and their probabilities?

**Checkpoint 4B.** Why is multiplying by $q$ again during evaluation wrong
for inverted dropout?


In [ ]:
drop = nn.Dropout(p=0.25)
values = torch.full((20000,), 6.0)
torch.manual_seed(SEED)
drop.train()
sample = drop(values)
sample_mean = sample.mean()
print("sample mean:", sample_mean.item(), "| expected:", 6.0)
assert torch.isclose(sample_mean, torch.tensor(6.0), atol=0.08, rtol=0.0)
drop.eval()
evaluated = drop(values)
assert torch.allclose(evaluated, values, atol=ATOL, rtol=RTOL)

## 5. Train mode and eval mode are different functions

`model.train()` recursively sets modules to training mode:

- BatchNorm uses batch statistics and updates running buffers;
- dropout samples a mask and rescales kept activations.

`model.eval()` recursively selects evaluation behavior:

- BatchNorm uses fixed running statistics without updating them;
- dropout is the identity.

Neither call enables or disables autograd. Use `torch.no_grad()` separately
for evaluation. A deterministic evaluation contract therefore requires both
`eval()` and a fixed model state; repeated calls on the same input should
match within named tolerances.

**Checkpoint 5A.** Why can evaluating one example in training mode fail for
BatchNorm even if dropout is absent?

**Checkpoint 5B.** If repeated evaluation logits differ, which mode/state
checks come first?


In [ ]:
class NormalizedMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 8)
        self.bn = nn.BatchNorm1d(8)
        self.drop = nn.Dropout(p=0.25)
        self.output = nn.Linear(8, 2)

    def forward(self, x):
        x = self.hidden(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.drop(x)
        return self.output(x)

torch.manual_seed(SEED)
mode_model = NormalizedMLP()
batch = torch.tensor([
    [-1.0, -1.0], [-1.0, 1.0], [1.0, -1.0], [1.0, 1.0],
    [-0.8, -1.2], [-1.1, 0.9], [0.9, -1.1], [1.2, 0.8],
])
mode_model.train()
torch.manual_seed(SEED)
train_a = mode_model(batch)
train_b = mode_model(batch)
mode_model.eval()
with torch.no_grad():
    eval_a = mode_model(batch)
    eval_b = mode_model(batch)
print("training calls equal:", torch.allclose(train_a, train_b, atol=ATOL, rtol=RTOL))
print("evaluation calls equal:", torch.allclose(eval_a, eval_b, atol=ATOL, rtol=RTOL))
assert not torch.allclose(train_a, train_b, atol=ATOL, rtol=RTOL)
assert torch.allclose(eval_a, eval_b, atol=ATOL, rtol=RTOL)

## 6. Worked combined training audit

**Scenario.** Validation accuracy changes each time it is measured, and
`running_mean` also changes during validation. The code uses
`with torch.no_grad(): logits = model(X_val)` but never calls
`model.eval()`.

**Diagnosis.**

1. `no_grad` stops graph recording only.
2. The model remains in training mode, so dropout samples new masks.
3. BatchNorm uses validation-batch statistics and mutates running buffers,
   leaking validation distribution into future state.
4. Fix: call `model.eval()` before the validation loop, wrap it in
   `no_grad`, and restore `model.train()` before the next training epoch.
5. Certification: snapshot running buffers before validation, require them
   unchanged afterward, and require repeated validation logits to agree.

**Checkpoint 6A.** Is the validation leak a parameter update? Explain.

**Checkpoint 6B.** What one assertion distinguishes a mode bug from
nondeterministic floating-point roundoff in this small CPU example?


In [ ]:
mode_model.eval()
mean_before = mode_model.bn.running_mean.clone()
var_before = mode_model.bn.running_var.clone()
with torch.no_grad():
    validation_1 = mode_model(batch)
    validation_2 = mode_model(batch)
assert torch.equal(mean_before, mode_model.bn.running_mean)
assert torch.equal(var_before, mode_model.bn.running_var)
assert torch.allclose(validation_1, validation_2, atol=ATOL, rtol=RTOL)
print("evaluation is deterministic and running buffers are unchanged")

## 7. Common pitfalls, exam connections, and C7 forward link

- **Wrong axis:** normalizing across features makes examples interact
  incorrectly. BatchNorm1d on $(N,D)$ normalizes each feature across $N$.
- **Parameter/buffer confusion:** $\gamma,\beta$ train by optimizer;
  running statistics update by module logic.
- **Mode-only misconception:** `eval()` does not turn off gradients;
  `no_grad()` does not select evaluation behavior.
- **Double dropout scaling:** inverted dropout already makes evaluation the
  identity.
- **Batch size one in training:** a feature variance cannot be estimated
  meaningfully; evaluation should use running statistics.

Round 1 items may ask which state changes, why repeated outputs differ, or
which formula preserves expectation. In C7, BatchNorm buffers and dropout
modes remain exactly these; only tensor shapes gain spatial axes and the
optimizer may target a selected subset of convolutional parameters.

**Checkpoint 7A.** Name every state category needed to resume training
faithfully for this combined model.

**Checkpoint 7B.** What must remain invariant during a correct evaluation
pass?


## Checkpoint answers

**1A.** Epsilon keeps the denominator positive; centered values are all zero.
**1B.** Means $(10)$, variances $(10)$, $\gamma(10)$, output $(32,10)$.

**2A.** Each sums the $N$ example contributions for one feature.
**2B.** Direct centered-input path, variance path, and mean path.

**3A.** `weight`, `bias`, `running_mean`, `running_var` (plus
`num_batches_tracked` as an integer buffer). **3B.** It is not a derivative-
trained quantity and has no gradient; BatchNorm's update rule owns it.

**4A.** $0$ with probability $1/4$ and $6/(3/4)=8$ with probability $3/4$.
**4B.** Training already scaled kept values by $1/q$; another $q$ would reduce
the expected evaluation activation.

**5A.** Training BatchNorm needs a batch variance and PyTorch rejects a
one-value-per-channel training batch. **5B.** Confirm `eval()`, dropout
identity, unchanged BatchNorm buffers, and fixed inputs/model state.

**6A.** No trainable parameter need move; BatchNorm buffers are persistent
nonparameter state. **6B.** Repeated eval-mode logits must satisfy
`allclose` while running buffers remain exactly equal.

**7A.** Model parameters, model buffers, optimizer state, and relevant random
generator state plus hyperparameters/data order. **7B.** Parameters and
buffers remain unchanged, gradients are not constructed, and repeated logits
match within fixed tolerances.
